# 語彙の豊富さを測る (TTR とその仲間)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/07_corpus_statistics/lexical_diversity.ipynb)

NLP クックブック 07-09 ／ コーパスの統計と探索

テキストがどれくらい多様な語彙を使っているかを数値化したい (作家の比較、学習者の習熟度の推定など)。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, math, re, urllib.request, zipfile
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

tagger = Tagger()

def tokens(text):
    return [w.feature.lemma or w.surface for line in text.split("\n") for w in tagger(line) if w.feature.pos1 != "補助記号"]

def mattr(toks, window=500):
    """移動平均 TTR: 一定幅の窓ごとの TTR を平均する (テキスト長の影響を受けにくい)"""
    if len(toks) < window:
        return len(set(toks)) / len(toks)
    return sum(len(set(toks[i:i + window])) for i in range(len(toks) - window + 1)) / (len(toks) - window + 1) / window

base = "https://www.aozora.gr.jp/cards/"
works = {"吾輩は猫である": "000148/files/789_ruby_5639.zip", "坊っちゃん": "000148/files/752_ruby_2438.zip",
         "走れメロス": "000035/files/1567_ruby_4948.zip", "銀河鉄道の夜": "000081/files/456_ruby_145.zip"}
print(f"{'':14}{'延べ':>8}{'異なり':>7}{'TTR':>7}{'Guiraud':>9}{'MATTR':>7}")
for title, path in works.items():
    t = tokens(aozora(base + path))
    N, V = len(t), len(set(t))
    print(f"{title:　<8}{N:>9}{V:>9}{V / N:>8.3f}{V / math.sqrt(N):>8.1f}{mattr(t):>8.3f}")

                    延べ    異なり    TTR  Guiraud  MATTR
吾輩は猫である　   187092    10729   0.057    24.8   0.412
坊っちゃん　　　    51786     4346   0.084    19.1   0.402
走れメロス　　　     5338     1154   0.216    15.8   0.417
銀河鉄道の夜　　    20633     1999   0.097    13.9   0.391


- 素の TTR (異なり語数 ÷ 延べ語数) はテキストが長いほど小さくなるため、長さの違うテキストの比較には使えない。MATTR や MTLD を使う。